In [0]:
from typing import Any

In [0]:
import logging

from py4j.java_gateway import JavaObject  # type: ignore
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    concat_ws,
    current_timestamp,
    from_utc_timestamp,
    lit,
    sha2,
    when,
    coalesce,
)

In [0]:
dbutils.widgets.text('curated_zone', 'None', '01-Curated_zone')
dbutils.widgets.text('catalog_landed', 'None', '02-Catalog_landed')
dbutils.widgets.text('reference_zone', 'None', '03-Reference_zone')
dbutils.widgets.text('run_date', 'None', '04-Run Date')
dbutils.widgets.text('schema', 'None', '05-Schema')
dbutils.widgets.text('env_prefix', 'None', '06-Environment')
curated_zone = dbutils.widgets.get('curated_zone')
catalog_landed = dbutils.widgets.get('catalog_landed')
reference_zone = dbutils.widgets.get('reference_zone')
run_date = dbutils.widgets.get('run_date')
schema = dbutils.widgets.get('schema')
env_prefix = dbutils.widgets.get('env_prefix')

import sys, os
relative_path = '../../helpers/lng_lnd'
path = os.path.abspath(relative_path)
if path not in sys.path:
  sys.path.append(path)
from landing_helpers import *
from landed_helpers import *
relative_path = '../../helpers/curated_helpers'
path = os.path.abspath(relative_path)
if path not in sys.path:
  sys.path.append(path)
from delta_table_creator import *
from extract import *
from transform import *
spark = SparkSession.builder.getOrCreate()


In [0]:
def load_curated_collateral_qca(
    spark, curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix
) -> None:
    args = [curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix]
    if all(item is not None for item in args) :
        date_filter = f"businessdate = {run_date!r}"

        # schema = (
        # "CollateralId string, SourceSystem string, PortfolioKey string,"
        # "PortfolioName string, IngestionTimestamp timestamp, BatchId string,"
        # "CollateralKey string, CollateralOldKey string, CollateralCode string, Comment string,"
        # "CollateralAmount decimal(19,2), CollateralCurrency string, CollateralStatusCode string,"
        # "CollateralStatus string,CollateralStatusChangeDate date,CollateralREStatusChangeDate date, StatusDate date, MortgageType string,# "MortgageTypeCode string,"
        # "AgreementKey string,"
        # "CategoryCode string, Category string, SubCategoryCode string, SubCategory string, OwnerPartyKey int, RealEstateKey int,"
        # " NonRealEstateKey int, NonRealEstateCollateralStatusCode string, NonRealEstateCollateralStatus string, OwnershipKey int,"
        # "RealTitleDescription string, OwnershipPercentage double,"
        # "OwnershipStatusCode string, OwnershipStatus string,"
        # "OwnershipStatusDate date, NotaryNumber string, NotaryDate date,"
        # "KtisisWayKey int,RealEstateCollateralKey string, RealEstateCollateralStatusCode string, RealEstateCollateralStatus string,"
        # "InitialLegalOrder int, LawyerPartyKey string,"
        # "JudgementNumber string, JudgementDate date, AppliedDate date,"
        # "RegistryNumber string, LandRegistry int,"
        # "TranscriptNumber string, TranscriptVolume string,"
        # "TranscriptSheet string, CreditorPartyKey string, CourtNumber int,"
        # "BusinessDate date"
        # )

        cluster_columns = ["BusinessDate"]

        curated_table_path = f"{curated_zone}.Collateral"

        # ================== Create Delta Table ==================
        logging.info("Creating Delta Table...")
        if spark.catalog.tableExists(curated_table_path) is False:
            create_delta_table(
                spark,
                curated_table_path,
                cluster_columns,
                schema,  # type: ignore
            )
        else:
            logging.info("Table already exists.")

        # ================== Delete From Delta Table ==================
        if spark.catalog.tableExists(curated_table_path) is True:
            delete_from_curated_table(
                    spark,
                    curated_table_path,
                    run_date,  # type: ignore
                    "qca",
                )        
        else:
            logging.info("Tables does not exist, no need to delete entries for business date!")


        # ========================== at_liens ==========================

        at_liens_owner_extra_filters = None

        if at_liens_owner_extra_filters is not None:
            at_liens_owner_extra_filters.append(date_filter)
            at_liens_final_filters = " AND ".join(
                at_liens_owner_extra_filters
            )
        else:
            at_liens_final_filters = date_filter

        at_liens_selected_fields = [
            "LIE_CODE",
            "LIE_ORDER",
            "LIE_LENDER",
        ]

        at_liens = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "qca_at_liens",
            at_liens_final_filters,
            at_liens_selected_fields,
        )

        at_liens = (
            at_liens.withColumnRenamed("LIE_CODE", "LIE_CODE_al")
            )

        # ========================== at_legal_collateral_liens ==========================

        at_legal_collateral_liens_owner_extra_filters = None

        if at_legal_collateral_liens_owner_extra_filters is not None:
            at_legal_collateral_liens_owner_extra_filters.append(date_filter)
            at_legal_collateral_liens_final_filters = " AND ".join(
                at_legal_collateral_liens_owner_extra_filters
            )
        else:
            at_legal_collateral_liens_final_filters = date_filter

        at_legal_collateral_liens_selected_fields = [
            "LIE_CODE",
            "LECO_CODE",
        ]

        at_legal_collateral_liens = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "qca_at_legal_collateral_liens",
            at_legal_collateral_liens_final_filters,
            at_legal_collateral_liens_selected_fields,
        )

        at_legal_collateral_liens = (
            at_legal_collateral_liens.withColumnRenamed("LIE_CODE", "LIE_CODE_alcl")
            )

        # ========================== at_legal_collaterals ==========================

        at_legal_collaterals_owner_extra_filters = None

        if at_legal_collaterals_owner_extra_filters is not None:
            at_legal_collaterals_owner_extra_filters.append(date_filter)
            at_legal_collaterals_final_filters = " AND ".join(
                at_legal_collaterals_owner_extra_filters
            )
        else:
            at_legal_collaterals_final_filters = date_filter

        at_legal_collaterals_selected_fields = [
            "LECO_CODE",
            "CASE_CODE",
            "LECO_S_ID",
            "LECO_EXTERNAL_ID",
            "LECO_DESC",
            "LECO_CURRENCY",
            "LECO_IS_ACTIVE",
            "LECO_STATUS_DATE",
            "LECO_ISSUE_DATE",
            "LECO_GUARANTED_VALUE",
        ]

        at_legal_collaterals = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "qca_at_legal_collaterals",
            at_legal_collaterals_final_filters,
            at_legal_collaterals_selected_fields,
        )

        at_legal_collaterals = (
            at_legal_collaterals.withColumnRenamed("CASE_CODE", "CASE_CODE_alc")
            .withColumnRenamed("LECO_CODE", "LECO_CODE_alc")
            )

        # ================== at_property_liens ==================

        at_property_liens_extra_filters = None

        if at_property_liens_extra_filters is not None:
            at_property_liens_extra_filters.append(date_filter)
            at_property_liens_final_filters = " AND ".join(
                at_property_liens_extra_filters
            )
        else:
            at_property_liens_final_filters = date_filter

        at_property_liens_selected_fields = [
            "LIE_CODE", 
            "PRO_CODE",
            ]

        at_property_liens = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "qca_at_property_liens",
            at_property_liens_final_filters,
            at_property_liens_selected_fields,
        )

        at_property_liens = (
            at_property_liens.withColumnRenamed("LIE_CODE", "LIE_CODE_apl")
            .withColumnRenamed("PRO_CODE", "PRO_CODE_apl")
            )

        # ========================== at_properties ==========================

        at_properties_owner_extra_filters = None

        if at_properties_owner_extra_filters is not None:
            at_properties_owner_extra_filters.append(date_filter)
            at_properties_final_filters = " AND ".join(
                at_properties_owner_extra_filters
            )
        else:
            at_properties_final_filters = date_filter

        at_properties_selected_fields = [
            "PRO_CODE",
            "REST_CODE",
            "PRO_ASSET_PERCENTAGE",
            "PRO_ASSET_TYPE_CODE",
            "CUST_CODE",
        ]

        at_properties = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "qca_at_properties",
            at_properties_final_filters,
            at_properties_selected_fields,
        )

        at_properties = (
                at_properties
                .withColumnRenamed("CUST_CODE", "CUST_CODE_pr")
                        )


        # ========================== at_real_estates ==========================

        at_real_estates_owner_extra_filters = None

        if at_real_estates_owner_extra_filters is not None:
            at_real_estates_owner_extra_filters.append(date_filter)
            at_real_estates_final_filters = " AND ".join(
                at_real_estates_owner_extra_filters
            )
        else:
            at_real_estates_final_filters = date_filter

        at_real_estates_selected_fields = [
            "REST_CODE",
            "REST_LAND_REGISTRY_NUMBER",
            "businessdate",
        ]

        at_real_estates = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "qca_at_real_estates",
            at_real_estates_final_filters,
            at_real_estates_selected_fields,
        )

        at_real_estates = (
                at_real_estates
                .withColumnRenamed("REST_CODE", "REST_CODE_atre")
                        )

        # ================== fs_cases_snap ==================

        fs_cases_snap_extra_filters = None

        if fs_cases_snap_extra_filters is not None:
            fs_cases_snap_extra_filters.append(date_filter)
            fs_cases_snap_final_filters = " AND ".join(fs_cases_snap_extra_filters)
        else:
            fs_cases_snap_final_filters = date_filter

        fs_cases_snap_selected_fields = [
            "CASE_CODE",
            "CASE_SOURCE",
            "SPET_CODE",
        ]

        fs_cases_snap = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "qca_fs_cases_snap",
            fs_cases_snap_final_filters,
            fs_cases_snap_selected_fields,
        )

        fs_cases_snap = (
            fs_cases_snap.withColumnRenamed("CASE_CODE", "CASE_CODE_fs")
            )

        # ================== dim_cases_static ==================

        dim_cases_static_extra_filters = None

        if dim_cases_static_extra_filters is not None:
            dim_cases_static_extra_filters.append(date_filter)
            dim_cases_static_final_filters = " AND ".join(
                dim_cases_static_extra_filters
            )
        else:
            dim_cases_static_final_filters = date_filter

        dim_cases_static_selected_fields = [
            "CASE_ACCOUNT_FOR_SEARCH",
            "CASE_CODE",
            "CASE_CONTRACT_ACCOUNT_KEY",
            "CASE_LEGAL_CONTRACT",
            "CUST_CODE",
        ]

        dim_cases_static = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "qca_dim_cases_static",
            dim_cases_static_final_filters,
            dim_cases_static_selected_fields,
        )

        dim_cases_static = (
            dim_cases_static.withColumnRenamed("CASE_CODE", "CASE_CODE_dcs")
            .withColumnRenamed("CUST_CODE", "CUST_CODE_dcs")
            )

        # ================== portfolio ==================

        portfolio_extra_filters = None

        if portfolio_extra_filters is not None:
            portfolio_extra_filters.append(date_filter)
            portfolio_final_filters = " AND ".join(portfolio_extra_filters)
        else:
            portfolio_final_filters = date_filter

        portfolio_selected_fields = [
            "PortfolioName",
            "Prefix",
            "portfolio_id_qca",
            "businessdate",
        ]

        portfolio = load_landed_portfolio_table(  # noqa: F841
            spark,
            curated_zone,  # type: ignore
            "portfolio",
            portfolio_final_filters,
            portfolio_selected_fields,
        )

        portfolio = (
            portfolio.withColumnRenamed("businessdate", "businessdate_port")
            )

        # ========================== Joins ==========================
        
        at_real_estates_at_properties_join_condition = (
            at_real_estates["REST_CODE_atre"] == at_properties["REST_CODE"]
        )

        at_properties_at_property_liens_join_condition = (
            at_properties["PRO_CODE"] == at_property_liens["PRO_CODE_apl"]
        )

        at_property_liens_at_liens_join_condition = (
            at_property_liens["LIE_CODE_apl"] == at_liens["LIE_CODE_al"]
        )

        at_liens_at_legal_collateral_liens_join_condition = (
            at_liens["LIE_CODE_al"] == at_legal_collateral_liens["LIE_CODE_alcl"]
        )

        at_legal_collateral_liens_at_legal_collaterals_join_condition = (
            at_legal_collateral_liens["LECO_CODE"] == at_legal_collaterals["LECO_CODE_alc"]
        )

        at_legal_collaterals_dim_cases_static_join_condition = (
            at_legal_collaterals["CASE_CODE_alc"] == dim_cases_static["CASE_CODE_dcs"]
        )

        dim_cases_static_fs_cases_snap_join_condition = (
            dim_cases_static["CASE_CODE_dcs"] == fs_cases_snap["CASE_CODE_fs"]
        )

        fs_cases_snap_portfolio_join_condition = (
            fs_cases_snap["SPET_CODE"] == portfolio["portfolio_id_qca"]
        )
        

        target_table = (
                at_real_estates
                .join(
                    at_properties,
                    at_real_estates_at_properties_join_condition,
                    "inner"
                )
                .join(
                    at_property_liens,
                    at_properties_at_property_liens_join_condition,
                    "left"
                )
                .join(
                    at_liens,
                    at_property_liens_at_liens_join_condition,
                    "left"
                )
                .join(
                    at_legal_collateral_liens,
                    at_liens_at_legal_collateral_liens_join_condition,
                    "left"
                )
                .join(
                    at_legal_collaterals,
                    at_legal_collateral_liens_at_legal_collaterals_join_condition,
                    "inner"
                )
                .join(
                    dim_cases_static,
                    at_legal_collaterals_dim_cases_static_join_condition,
                    "inner"
                )
                .join(
                    fs_cases_snap,
                    dim_cases_static_fs_cases_snap_join_condition,
                    "left"
                )
                .join(
                    portfolio,
                    fs_cases_snap_portfolio_join_condition,
                    "left"
                )
            )

        target_table = target_table.withColumn(
                "AgreementKey",
                when(
                    (fs_cases_snap.CASE_SOURCE).like("C%"),
                    dim_cases_static.CASE_CONTRACT_ACCOUNT_KEY,
                ).otherwise(dim_cases_static.CASE_LEGAL_CONTRACT),
        ).withColumn(
                "collateral_code",
                concat_ws(
                    "",
                    coalesce(portfolio["prefix"], lit("")),
                    coalesce(at_legal_collaterals["LECO_CODE_alc"], lit(""))
                )
        )

        primary_key_fields = [
            target_table.LECO_CODE_alc,
            target_table.AgreementKey,
            target_table.PRO_CODE,
            lit("qca"),
        ]

        target_table = target_table.withColumn(
            "CollateralId", sha2(concat_ws(";", *primary_key_fields), 256)
        ).withColumn(
            "CollateralStatusCode",
            when(target_table.LECO_IS_ACTIVE == True, lit("Ενεργή"))
            .when(target_table.LECO_IS_ACTIVE == False, lit("Ανενεργή"))
            .otherwise(lit(None))
        )


        # ================== result_df ==================
        result_df = target_table.select(
            target_table.CollateralId,
            lit("qca").alias("SourceSystem"),
            target_table.portfolio_id_qca.cast("string").alias("PortfolioKey"), 
            target_table.PortfolioName.cast("string").alias("PortfolioName"),
            lit(from_utc_timestamp(current_timestamp(), "Europe/Athens")).alias(
                "IngestionTimestamp"
            ),
            lit(None).alias("BatchID"),
            target_table.LECO_S_ID.cast("string").alias("CollateralKey"), 
            lit(None).cast("string").alias("CollateralOldKey"),
            target_table.collateral_code.alias("CollateralCode"), 
            target_table.LECO_DESC.alias("Comment"),
            target_table.LECO_GUARANTED_VALUE.cast("decimal(19,2)").alias("CollateralAmount"),
            target_table.LECO_CURRENCY.alias("CollateralCurrency"),
            target_table.CollateralStatusCode.cast("string").alias("CollateralStatusCode"), 
            target_table.CollateralStatusCode.cast("string").alias("CollateralStatus"),   
            target_table.LECO_STATUS_DATE.cast("date").alias("CollateralStatusChangeDate"), #LECO_STATUS_DATE has no values from landed
            lit(None).cast("date").alias("CollateralREStatusChangeDate"),
            lit(None).cast("date").alias("StatusDate"),
            lit(None).alias("MortgageTypeCode"),
            lit(None).alias("MortgageType"),
            target_table.AgreementKey.alias("AgreementKey"),
            lit(None).alias("CategoryCode"),
            lit(None).alias("Category"),
            lit(None).alias("SubCategoryCode"),
            lit(None).alias("SubCategory"),
            target_table.CUST_CODE_dcs.cast("int").alias("OwnerPartyKey"), 
            target_table.REST_CODE.cast("int").alias("RealEstateKey"), 
            lit(None).cast("int").alias("NonRealEstateKey"),
            lit(None).cast("string").alias("NonRealEstateCollateralStatusCode"),
            lit(None).cast("string").alias("NonRealEstateCollateralStatus"),
            target_table.CUST_CODE_pr.alias("OwnershipKey"), 
            lit(None).alias("RealTitleDescription"),
            target_table.PRO_ASSET_PERCENTAGE.cast("double").alias("OwnershipPercentage"),
            lit(None).cast("string").alias("OwnershipStatusCode"),  
            lit(None).alias("OwnershipStatus"),
            lit(None).alias("OwnershipStatusDate"),
            lit(None).alias("NotaryNumber"),
            lit(None).alias("NotaryDate"),
            lit(None).cast("int").alias("KtisisWayKey"),
            target_table.PRO_CODE.cast("string").alias("RealEstateCollateralKey"),
            lit(None).cast("string").alias(
                "RealEstateCollateralStatusCode"
            ),
            lit(None).cast("string").alias(
                "RealEstateCollateralStatus"
            ),
            target_table.LIE_ORDER.cast("int").alias("InitialLegalOrder"), 
            lit(None).cast("string").alias("LawyerPartyKey"),
            lit(None).alias("JudgementNumber"),
            lit(None).alias("JudgementDate"),
            target_table.LECO_ISSUE_DATE.cast("date").alias("AppliedDate"),
            target_table.REST_LAND_REGISTRY_NUMBER.alias("RegistryNumber"), 
            lit(None).alias("LandRegistry"), 
            lit(None).alias("TranscriptNumber"),
            lit(None).alias("TranscriptVolume"),
            lit(None).alias("TranscriptSheet"),
            lit(None).cast("string").alias("CreditorPartyKey"), 
            lit(None).alias("CourtNumber"),
            target_table.businessdate.cast("date").alias("BusinessDate"), 
        ).distinct()

        # ========================== Append ==========================

        result_df.write.format("delta").clusterBy("BusinessDate").mode("append").saveAsTable(
            curated_table_path
        )
    else:
        null_args = [
            arg_name
            for arg_name, arg in args
            if arg
            in [
                curated_zone,
                catalog_landed,
                reference_zone,
                run_date,
                schema,
                env_prefix,
            ]
            and arg is None
        ]

        for arg_name in null_args:
            logging.error(f"No {arg_name} specified!")

    logging.info("We did it!")


In [0]:
load_curated_collateral_qca(spark, curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix)